In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder.appName("Spark DataFrames").getOrCreate()


In [0]:
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]


In [0]:
bronze=spark.createDataFrame(data,columns)
display(bronze)

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
bronze.write.format("delta").mode("append").save("/Volumes/medallion/bronze/bronze_data")

silver


In [0]:
df=spark.read.format("delta").load("/Volumes/medallion/bronze/bronze_data")

df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df = df.withColumn("date",col("date").cast("date"))
df=df.withColumn("amount",col("amount").cast("int"))

In [0]:
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


order_id,customer_id,product,category,city,date,amount,quantity,rn
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1,2
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2,2
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,1


In [0]:


# Window: customer + product
window_spec = Window.partitionBy("customer_id", "product") \
                    .orderBy(col("date").desc())

# Rank
df_with_rank = df.withColumn("rn", row_number().over(window_spec))

# Keep latest per product
final_df = df_with_rank.filter(col("rn") == 1).drop("rn")

final_df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+



In [0]:
final_df=final_df.filter(col("amount")>0)
final_df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
final_df=final_df.fillna({
    'city':'unkown'
})
final_df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,unkown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
final_df.write.format("delta").mode("append").save("/Volumes/medallion/silver/silver_data")

gold

In [0]:
gold_df=spark.read.format("delta").load("/Volumes/medallion/silver/silver_data")
gold_df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,unkown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
gold_df.write.format("delta").mode("append").save("/Volumes/medallion/gold/gold_data")


In [0]:
final_df=spark.read.format("delta").load("/Volumes/medallion/gold/gold_data")

In [0]:
final_df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,unkown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
#Tasks:
##Aggregations
#Total sales by product
#Total sales by category
#Total sales by city

##Customer Metrics
#Total orders per customer
#Total spending per customer

##Top Analysis
#Top selling product
#Top customer


In [0]:
final_df.groupBy("product").agg(sum("amount")).show()

+----------+-----------+
|   product|sum(amount)|
+----------+-----------+
|    Tablet|      22000|
|    Mobile|      33000|
|     Watch|       8000|
|    Laptop|     105000|
|Headphones|       3000|
+----------+-----------+



In [0]:
final_df.groupBy("category").agg(sum("amount")).show()

+-----------+-----------+
|   category|sum(amount)|
+-----------+-----------+
|Electronics|     160000|
|Accessories|      11000|
+-----------+-----------+



In [0]:
final_df.groupBy("city").agg(sum("amount")).show()

+---------+-----------+
|     city|sum(amount)|
+---------+-----------+
|    Delhi|      55000|
|  Chennai|      33000|
|Hyderabad|      72000|
|   unkown|       3000|
|   Mumbai|       8000|
+---------+-----------+



In [0]:
final_df.groupBy("customer_id").agg(count("order_id")).show()

+-----------+---------------+
|customer_id|count(order_id)|
+-----------+---------------+
|       C005|              1|
|       C004|              1|
|       C003|              1|
|       C001|              2|
|       C002|              1|
|       C007|              1|
+-----------+---------------+



In [0]:
final_df.groupBy("customer_id").agg(sum(col('amount')*col("quantity"))).show()

+-----------+------------------------+
|customer_id|sum((amount * quantity))|
+-----------+------------------------+
|       C005|                    3000|
|       C004|                    8000|
|       C003|                   55000|
|       C001|                   72000|
|       C002|                   18000|
|       C007|                   30000|
+-----------+------------------------+



In [0]:
final_df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,unkown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
#Top selling product
#final_df.groupBy("product").agg(sum("amount")).orderBy(desc("sum(amount)")).show(1)
final_df.orderBy(desc("quantity")).groupBy("product").count().show(1)
#Top customer
final_df.groupBy("customer_id").agg(sum(col('amount')*col("quantity"))).orderBy(desc("sum((amount * quantity))")).show(1)

+-------+-----+
|product|count|
+-------+-----+
| Tablet|    1|
+-------+-----+
only showing top 1 row
+-----------+------------------------+
|customer_id|sum((amount * quantity))|
+-----------+------------------------+
|       C001|                   72000|
+-----------+------------------------+
only showing top 1 row


In [0]:
# Step 1: Aggregate
df_prod = final_df.groupBy("product") \
    .agg(sum("amount").alias("total_sales"))

# Step 2: Window for ranking
window_spec = Window.orderBy(desc("total_sales"))

# Step 3: Rank
df_ranked = df_prod.withColumn("rank", dense_rank().over(window_spec))

# Step 4: Get top product
df_ranked.filter("rank = 1").show()

+-------+-----------+----+
|product|total_sales|rank|
+-------+-----------+----+
| Laptop|     105000|   1|
+-------+-----------+----+



In [0]:
from pyspark.sql.functions import expr

# Step 1: Calculate total spend
df_cust = final_df.groupBy("customer_id") \
    .agg(expr("sum(amount * quantity) as total_spent"))

# Step 2: Window ranking
window_spec = Window.orderBy(desc("total_spent"))

df_ranked = df_cust.withColumn("rank", dense_rank().over(window_spec))

# Step 3: Top customer
df_ranked.filter("rank = 1").show()

+-----------+-----------+----+
|customer_id|total_spent|rank|
+-----------+-----------+----+
|       C001|      72000|   1|
+-----------+-----------+----+

